# ML-10 — Content Action Playbook

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/huydang2006/flyrank-ML-internship/blob/main/work/notebooks/w07_action_playbook.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Ranked actions + reason codes

*The queue: what to do first, and why, in words a human trusts.*

Review the first 50 rows per cycle. Keep the observed reason code, supporting signals, suggested action, confidence note, and model signal note together. The score prioritizes review; it is not a causal explanation.

In [1]:
from pathlib import Path
import subprocess
import sys

ROOT = Path.cwd()
if ROOT.name == 'notebooks' and ROOT.parent.name == 'work':
    ROOT = ROOT.parent.parent
result = subprocess.run(
    [sys.executable, str(ROOT / 'work' / 'scripts' / 'ml10_playbook.py')],
    cwd=ROOT, text=True, capture_output=True,
)
print(result.stdout)
print(result.stderr)
if result.returncode != 0:
    raise RuntimeError(f'ml10_playbook.py failed with exit code {result.returncode}')


Wrote D:\EP16\flyrank-ML-internship\work\outputs\ml10_action_queue_public.csv
Wrote D:\EP16\flyrank-ML-internship\work\outputs\ml10_action_playbook.md




## 2. Intended use and limits

*Who uses this, for what — and where it stops being valid.*

This is a portfolio review aid for an SEO/content specialist. Default capacity is 50 pages per cycle. It is not an automated publishing system, a hiring tool, or evidence that an edit will improve traffic or CTR. Low-confidence and contradictory rows remain human-review candidates.

In [2]:
import csv
queue_path = ROOT / 'work' / 'outputs' / 'ml10_action_queue_public.csv'
with queue_path.open(newline='', encoding='utf-8') as handle:
    queue = list(csv.DictReader(handle))
print(f'Public queue rows: {len(queue):,}')
print(f'Paper-facing fields: {list(queue[0])}')


Public queue rows: 2,904
Paper-facing fields: ['rank', 'model_score', 'model_rank', 'baseline_score', 'baseline_rank', 'primary_reason_code', 'supporting_reason_codes', 'suggested_action', 'confidence_note', 'model_signal_codes', 'model_signal_note', 'impressions_90d', 'ctr_feature', 'expected_ctr_feature', 'error_type']


## 3. Human review + the no-go list

*What a person must check before acting. What should never be automated.*

1. Confirm page and query context in the source system.
2. Check impression volume and CTR denominator.
3. Compare CTR with the position-tier reference.
4. Inspect title, snippet, intent match, and technical status.
5. Record the chosen action, evidence, and reviewer confidence.
6. Re-measure after a defined observation window.

Never auto-publish edits, infer causality or intent from a score, use the queue for hiring or individual performance evaluation, or act below the minimum denominator.

In [3]:
hidden_fields = {'content_hash_id', 'client_hash_id'}
assert not hidden_fields.intersection(queue[0])
print('Public-safety identifier check: PASS')


Public-safety identifier check: PASS


## 4. Monitoring / retrain triggers

*What would tell you the recommendations went stale?*

Recheck Precision@50 and base rate each evaluation period. Review thresholds when the base rate or top-50 precision leaves the validated range. Monitor impressions, CTR, position, missingness, and action mix for drift. Retrain only after a new time-aware evaluation preserves feature/label separation.

In [4]:
import json
metrics = json.loads((ROOT / 'work' / 'outputs' / 'ml09_validation_metrics.json').read_text())
grouped = [row['model']['precision_at_50'] for row in metrics['grouped_validation']]
time_aware = [row['model']['precision_at_50'] for row in metrics['time_aware_validation']]
print(f'Validated grouped P@50 range: {min(grouped):.2f}-{max(grouped):.2f}')
print(f'Validated time-aware P@50 range: {min(time_aware):.2f}-{max(time_aware):.2f}')


Validated grouped P@50 range: 0.88-0.96
Validated time-aware P@50 range: 0.72-0.90


## 5. Exports for the paper

*Write the queue (and any figures you want to reuse) to work/outputs/ — your paper builds on these files.*

The reproducible outputs are the public queue, the action playbook, and ML-09 validation figures in `work/figures/`.

In [5]:
for path in (
    ROOT / 'work' / 'outputs' / 'ml10_action_queue_public.csv',
    ROOT / 'work' / 'outputs' / 'ml10_action_playbook.md',
    ROOT / 'work' / 'figures' / 'ml09_validation_precision_at_50.png',
    ROOT / 'work' / 'figures' / 'ml09_threshold_sensitivity.png',
):
    print(f'{path.relative_to(ROOT)}: {path.exists()}')


work\outputs\ml10_action_queue_public.csv: True
work\outputs\ml10_action_playbook.md: True
work\figures\ml09_validation_precision_at_50.png: True
work\figures\ml09_threshold_sensitivity.png: True


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.